<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part D: Deep Learning Approaches</h2>
<h2>Notebook D04: Forecasting with Transformers</h2>
</div>

Both architectures so far impose a fixed structure on how information moves through the window. Recurrence
walks it in order, so hour 1 reaches hour 168 only by being carried through the 167 steps in between.
Convolution slides a filter, so reaching back a week takes either many layers or aggressive dilation.

**Attention** removes the structure. Every position is compared directly with every other, and the model
learns which comparisons matter. Reaching from hour 168 back to hour 1 is a single operation rather than a
journey.

This notebook builds that mechanism from the arithmetic up, uses it to forecast, and then looks inside to
see what the trained model actually chose to attend to.

> This notebook needs PyTorch: `uv sync --group dl`.

---

**Contents**

1. [Imports and the Same Windows](#1.-Imports-and-the-Same-Windows)
2. [Attention Instead of Structure](#2.-Attention-Instead-of-Structure)
3. [Scaled Dot-Product Attention](#3.-Scaled-Dot-Product-Attention)
4. [Position Has to Be Added Back](#4.-Position-Has-to-Be-Added-Back)
5. [A Transformer Forecaster](#5.-A-Transformer-Forecaster)
6. [Looking Inside the Attention](#6.-Looking-Inside-the-Attention)
7. [The Comparison](#7.-The-Comparison)
8. [When a Transformer Is Worth It](#8.-When-a-Transformer-Is-Worth-It)

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="1.-Imports-and-the-Same-Windows">1. Imports and the Same Windows</h3>
</div>

In [ ]:
import importlib.util
import math
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error

import nb_config

sns.set_theme(style="whitegrid")

TORCH_AVAILABLE = importlib.util.find_spec("torch") is not None

if TORCH_AVAILABLE:
    import torch
    from torch import nn
    from torch.nn import functional as F
    from torch.utils.data import DataLoader, TensorDataset

    torch.set_num_threads(1)
    print(f"PyTorch {torch.__version__}")
else:
    print("PyTorch is not installed. Run 'uv sync --group dl' to follow this notebook.")

The same windows as Notebooks [D02](./D02_Recurrent_networks.ipynb) and
[D03](./D03_Convolutional_networks.ipynb): a week of hourly Austrian load in, the next 24 hours out.

In [ ]:
ops = pd.read_parquet(nb_config.OPS_15M_PATH)

load = (
    ops[(ops["country"] == "AT") & (ops["measure"] == "actual_entsoe_transparency")]["value"]
    .tz_convert(None)
    .resample("h").mean()
    .dropna()
    .asfreq("h")
    .loc["2016-01-01":"2019-12-31"]
)

LOOKBACK, HORIZON = 168, 24

# ── Run size ──────────────────────────────────────────────────────────────────
# Attention costs O(length^2), which makes this the most expensive notebook in
# the course. FULL_RUN trains on every window and reproduces the numbers quoted
# in the text; the default trains on every third window and finishes in a few
# minutes, which is what a workshop session has time for.
FULL_RUN = False

TRAIN_STRIDE = 1 if FULL_RUN else 3
EPOCHS = 6 if FULL_RUN else 8
LEARNING_RATE = 1e-3 if FULL_RUN else 3e-3

values = load.values.astype(np.float32)
n_observations = len(values)
TEST_HOURS = VALIDATION_HOURS = 24 * 90
train_end = n_observations - TEST_HOURS - VALIDATION_HOURS

mean, std = values[:train_end].mean(), values[:train_end].std()
scaled = (values - mean) / std


def make_windows(scaled, start, stop, stride=1):
    positions = range(start, stop, stride)
    inputs = np.stack([scaled[t - LOOKBACK:t] for t in positions])
    targets = np.stack([scaled[t:t + HORIZON] for t in positions])
    return torch.tensor(inputs)[:, :, None], torch.tensor(targets)


def to_original_units(scaled_values):
    return np.asarray(scaled_values) * std + mean


def score(predictions, targets):
    return mean_absolute_error(
        to_original_units(targets).ravel(), to_original_units(predictions).ravel()
    )


if TORCH_AVAILABLE:
    X_train, y_train = make_windows(scaled, LOOKBACK, train_end - HORIZON, TRAIN_STRIDE)
    X_validation, y_validation = make_windows(
        scaled, train_end, train_end + VALIDATION_HOURS - HORIZON
    )
    X_test, y_test = make_windows(
        scaled, train_end + VALIDATION_HOURS, n_observations - HORIZON
    )

    naive_positions = range(train_end + VALIDATION_HOURS, n_observations - HORIZON)
    naive_forecast = np.stack([values[t - 24:t - 24 + HORIZON] for t in naive_positions])
    NAIVE_MAE = mean_absolute_error(
        to_original_units(y_test.numpy()).ravel(), naive_forecast.ravel()
    )

    # Carried over from D02 and D03
    PREVIOUS = {
        "LSTM (D02)": (368.6, 124),
        "LSTM stacked (D02)": (350.4, 295),
        "Simple CNN (D03)": (291.6, 22),
        "TCN (D03)": (260.6, 283),
    }

    print(f"{'FULL' if FULL_RUN else 'WORKSHOP'} run: stride {TRAIN_STRIDE}, "
          f"{EPOCHS} epochs, learning rate {LEARNING_RATE}")
    print(f"train {tuple(X_train.shape)}   test {tuple(X_test.shape)}")
    print(f"Naive baseline: {NAIVE_MAE:.1f} MW")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="2.-Attention-Instead-of-Structure">2. Attention Instead of Structure</h3>
</div>

The distance information has to travel is the thread running through Part D.

| Architecture | Steps from hour 1 to hour 168 |
|---|---|
| **MLP** (D01) | 1, but every position has its own weights and no shared rule |
| **RNN / LSTM** (D02) | 168 sequential state updates |
| **TCN** (D03) | 14 layers, by design of the dilation schedule |
| **Attention** | 1 |

Attention computes, for every pair of positions, how relevant one is to the other, and mixes the sequence
accordingly. Nothing about the architecture privileges nearby positions: hour 1 and hour 167 are the same
distance apart as hour 166 and hour 167.

That is the strength and the catch. The model is free to learn any dependency pattern, which means it has
to **learn** the ones that recurrence and convolution were built to assume. Freedom costs data.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="3.-Scaled-Dot-Product-Attention">3. Scaled Dot-Product Attention</h3>
</div>

Each position produces three vectors by linear projection:

- a **query**: what this position is looking for;
- a **key**: what this position offers;
- a **value**: what it passes on if selected.

Compare every query with every key by dot product, scale by $\sqrt{d}$ to keep the numbers in a sensible
range, softmax each row into weights that sum to 1, and use those weights to average the values:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right)V$$

The matrix inside the softmax is $L \times L$: for a 168-hour window, 28,224 pairwise comparisons per
head. That quadratic cost is the price of the direct connection, and it is why long-context transformers
are a research field of their own.

**Multi-head** attention runs several of these in parallel on slices of the representation, so one head can
track the daily cycle while another tracks the weekly one.

In [ ]:
def scaled_dot_product_attention(queries, keys, values):
    """Attention from first principles, for one head.

    queries, keys, values: (length, dimension)
    Returns the mixed values and the (length, length) weight matrix.
    """
    dimension = queries.shape[-1]
    scores = queries @ keys.transpose(-2, -1) / math.sqrt(dimension)
    weights = torch.softmax(scores, dim=-1)
    return weights @ values, weights


if TORCH_AVAILABLE:
    torch.manual_seed(0)

    # A toy sequence of 6 steps, each described by 4 numbers
    length, dimension = 6, 4
    example = torch.randn(length, dimension)

    mixed, weights = scaled_dot_product_attention(example, example, example)

    print("attention weight matrix (rows sum to 1):")
    print(np.round(weights.numpy(), 2))
    print()
    print(f"row sums: {weights.sum(dim=-1).numpy().round(3)}")

Each row of that matrix says where one position looked, and each row sums to 1 because of the softmax. The
weights are the entire mechanism: everything a transformer does to a sequence is decided by this matrix,
and section 6 reads it back out of a trained model.

Note that the same tensor was used as queries, keys and values. That is **self**-attention: the sequence
attending to itself. In an encoder-decoder transformer, the decoder's queries attend to the encoder's keys
and values instead, which is how translation models connect two different sequences.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="4.-Position-Has-to-Be-Added-Back">4. Position Has to Be Added Back</h3>
</div>

There is a problem with what we just built, and stating it precisely matters.

Attention computes a weighted average over positions, and an average does not care about order. Nothing in
the equations above refers to *when* a value occurred. The exact property this gives a stack of attention
layers is **permutation equivariance**: permute the input and every output permutes with it, unchanged in
value.

$$\text{encoder}(X)_{\pi} = \text{encoder}(X_{\pi})$$

In other words the layer sees a **set**, not a sequence. It can tell you what values are present and how
they relate, and nothing whatever about their order or their spacing.

That is the thing to test, and the test has to be set up carefully: comparing `f(x)` with `f(shuffled x)`
for our model would prove nothing, because the head reads the final position and shuffling puts a
different value there. The honest comparison is between encoding first and permuting after, against
permuting first and encoding after.

In [ ]:
class PositionalEncoding(nn.Module):
    """Add a fixed sine and cosine signature to each position."""

    def __init__(self, dimension, max_length=2000):
        super().__init__()
        encoding = torch.zeros(max_length, dimension)
        position = torch.arange(max_length).unsqueeze(1).float()
        frequency = torch.exp(
            torch.arange(0, dimension, 2).float() * (-math.log(10000.0) / dimension)
        )
        encoding[:, 0::2] = torch.sin(position * frequency)
        encoding[:, 1::2] = torch.cos(position * frequency)
        self.register_buffer("encoding", encoding)

    def forward(self, x):
        return x + self.encoding[:x.size(1)].unsqueeze(0)


class TransformerForecaster(nn.Module):
    """Encoder-only transformer with a regression head."""

    def __init__(self, d_model=32, n_heads=4, n_layers=2, feedforward=64,
                 dropout=0.1, use_positional_encoding=True, horizon=HORIZON):
        super().__init__()
        self.project = nn.Linear(1, d_model)
        self.use_positional_encoding = use_positional_encoding
        self.positional = PositionalEncoding(d_model)

        layer = nn.TransformerEncoderLayer(
            d_model, n_heads, feedforward, dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.head = nn.Linear(d_model, horizon)

    def forward(self, x):
        h = self.project(x)
        if self.use_positional_encoding:
            h = self.positional(h)
        h = self.encoder(h)
        return self.head(h[:, -1])      # the last position, having attended to all others


if TORCH_AVAILABLE:
    torch.manual_seed(0)
    window = X_test[:1]
    order = torch.randperm(LOOKBACK)

    for label, use_encoding in [("without positional encoding", False),
                                ("with positional encoding", True)]:
        model = TransformerForecaster(use_positional_encoding=use_encoding).eval()

        with torch.no_grad():
            embedded = model.project(window)
            if use_encoding:
                embedded = model.positional(embedded)
                shuffled = model.positional(model.project(window[:, order]))
            else:
                shuffled = model.project(window[:, order])

            encode_then_shuffle = model.encoder(embedded)[:, order]
            shuffle_then_encode = model.encoder(shuffled)

        difference = (encode_then_shuffle - shuffle_then_encode).abs().max().item()
        print(f"{label:<30} max difference = {difference:.7f}")

Without positional encoding the difference is zero to floating-point precision, around $10^{-7}$. The two
routes are the same computation: the encoder genuinely cannot distinguish a week of load from the same 168
values in a different order. Adding the positional encoding breaks that equality outright, and the
difference jumps by seven orders of magnitude.

It is worth being clear about what the architecture does and does not give you for free. Reading the final
position, as our head does, smuggles in a little order information: the model can tell which value is most
recent. What it cannot do without help is tell whether a value occurred one hour ago or a hundred, which
is exactly what a forecast of electricity load depends on.

The encoding itself is a fixed pattern of sines and cosines at geometrically spaced frequencies, added to
the input representation. Each position gets a distinctive signature, and because the frequencies differ,
the *difference* between two signatures encodes how far apart they are. Nothing is learned: it is a
coordinate system for time, handed to the model.

For time series there is an obvious refinement the slides note. Rather than encoding "168 steps from the
start of the window", encode the actual calendar: hour of day, day of week. That is the same information
the calendar features of Notebook [C01](./C01_Feature_engineering.ipynb) carried, reaching a transformer
through the door built for it.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="5.-A-Transformer-Forecaster">5. A Transformer Forecaster</h3>
</div>

The model above is **encoder-only**: stack self-attention layers over the input window, take the
representation at the final position, and map it to 24 outputs with a linear head.

That is one of three arrangements, and worth distinguishing:

| Architecture | How it forecasts | Typical use |
|---|---|---|
| **Encoder-only** | Read the window, regress the horizon in one shot | What we build here; simple and fast |
| **Decoder-only** | Generate the horizon one step at a time, each conditioned on the last | GPT-style; natural for long horizons |
| **Encoder-decoder** | Encode the history, decode the horizon attending back to it | Temporal Fusion Transformer and similar |

Encoder-only suits a fixed 24-hour horizon and avoids the error accumulation that step-by-step generation
brings.

> **The next cell trains the model and takes a few minutes.**

In [ ]:
def train_model(build, epochs=None, batch_size=256, learning_rate=None, seed=0):
    """The training loop from D02 and D03, unchanged."""
    epochs = EPOCHS if epochs is None else epochs
    learning_rate = LEARNING_RATE if learning_rate is None else learning_rate

    torch.manual_seed(seed)
    generator = torch.Generator().manual_seed(seed)

    model = build()
    optimiser = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.MSELoss()
    loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size, shuffle=True, generator=generator,
    )

    started = time.time()
    best = {"validation_mae": np.inf, "epoch": 0, "weights": None}

    for epoch in range(epochs):
        model.train()
        for batch_X, batch_y in loader:
            optimiser.zero_grad()
            loss_function(model(batch_X), batch_y).backward()
            optimiser.step()

        model.eval()
        with torch.no_grad():
            validation_mae = score(model(X_validation).numpy(), y_validation.numpy())

        if validation_mae < best["validation_mae"]:
            best = {"validation_mae": validation_mae, "epoch": epoch,
                    "weights": {k: v.clone() for k, v in model.state_dict().items()}}

    model.load_state_dict(best["weights"])
    model.eval()
    with torch.no_grad():
        predictions = model(X_test).numpy()

    return {
        "model": model, "predictions": predictions,
        "test_mae": score(predictions, y_test.numpy()),
        "validation_mae": best["validation_mae"],
        "parameters": sum(p.numel() for p in model.parameters()),
        "seconds": time.time() - started,
    }


if TORCH_AVAILABLE:
    transformer = train_model(TransformerForecaster)
    print(f"Transformer: test MAE {transformer['test_mae']:.1f} MW, "
          f"{transformer['parameters']:,} parameters, {transformer['seconds']:.0f}s")

**Exercise.** Section 4 showed that without positional encoding the encoder cannot distinguish a week of load from the same values shuffled. Now measure what that costs: train `TransformerForecaster(use_positional_encoding=False)` and compare its test MAE against both the encoded model and the naive baseline. Given that the head reads the final position, and so always knows which value is most recent, why is the damage as large as it is?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="6.-Looking-Inside-the-Attention">6. Looking Inside the Attention</h3>
</div>

Attention weights are the one part of a deep model that can be read directly. The final position's row of
the attention matrix says which hours of the past week the model consulted in order to forecast the next
day.

This is the closest thing Part D has offered to the feature importances of Notebook
[C01](./C01_Feature_engineering.ipynb), and it is worth treating with the same caution: attention weights
show where the model looked, which is not quite the same as what caused its answer.

In [ ]:
if TORCH_AVAILABLE:
    model = transformer["model"]
    sample = X_test[:256]

    # Re-run the first encoder layer's attention, asking for the weights
    with torch.no_grad():
        embedded = model.positional(model.project(sample))
        first_layer = model.encoder.layers[0]
        _, attention_weights = first_layer.self_attn(
            embedded, embedded, embedded, need_weights=True, average_attn_weights=True
        )

    # Where did the final position look, averaged over the sample?
    final_row = attention_weights[:, -1, :].mean(dim=0).numpy()
    hours_back = np.arange(LOOKBACK, 0, -1)

    fig, axes = plt.subplots(2, 1, figsize=(13, 7))

    axes[0].plot(hours_back, final_row, color="steelblue", linewidth=1.0)
    for marker, label in [(24, "1 day"), (48, "2 days"), (168, "1 week")]:
        axes[0].axvline(marker, color="crimson", linestyle="--", linewidth=1.0, alpha=0.7)
        axes[0].text(marker, final_row.max() * 0.95, f" {label}", fontsize=9, color="crimson")
    axes[0].invert_xaxis()
    axes[0].set_title("Where the final position attends, averaged over the test set",
                      fontsize=13, fontweight="bold")
    axes[0].set_xlabel("Hours before the forecast")
    axes[0].set_ylabel("Attention weight")

    im = axes[1].imshow(attention_weights[0].numpy(), aspect="auto", cmap="viridis")
    axes[1].set_title("Full attention matrix for one window", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Attends to (position in window)")
    axes[1].set_ylabel("From (position in window)")
    fig.colorbar(im, ax=axes[1], label="weight")

    axes[0].grid(linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

This is not the picture the diagram in a transformer tutorial leads you to expect, and it is worth looking
at carefully.

**The attention is almost flat.** Spread evenly across 168 positions, every weight would be 1/168 =
0.0060. The observed mean is 0.0060, and the spread around it is small. The model is not concentrating on
a handful of informative hours; it is close to averaging the window.

**Where it does deviate, it is not where domain knowledge would predict.** The position 24 hours back, the
same hour yesterday, which every other model in Part D leaned on and which the naive baseline consists
entirely of, receives about 0.0007: an order of magnitude *below* uniform. The heaviest weights sit around
150 hours back, which corresponds to nothing in particular.

**And yet the model forecasts well** — better than either recurrent model in D02. Whatever skill it has is
not visible in this map.

The resolution is that a transformer layer is not only its attention. Each position's representation
passes through a residual connection and a feed-forward network, so the final position carries its own
recent values forward regardless of what the attention does, and the head reads that. Attention is one
route through the layer, not the whole of it.

The lesson generalises well beyond this notebook. **Attention weights show where information was gathered
from, not what caused the answer.** They are routinely presented as explanations, and there is a
substantial literature arguing that they should not be. Treat a map like this as a hypothesis worth
testing by other means, exactly as Notebook [C01](./C01_Feature_engineering.ipynb) treated feature
importances.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="7.-The-Comparison">7. The Comparison</h3>
</div>

Every sequence model in Part D, on identical windows and the same test period.

In [ ]:
if TORCH_AVAILABLE:
    rows = [{"Model": "Naive (repeat yesterday)", "Test MAE": NAIVE_MAE, "Seconds": 0.0}]
    rows += [{"Model": name, "Test MAE": mae, "Seconds": float(seconds)}
             for name, (mae, seconds) in PREVIOUS.items()]
    rows.append({"Model": "Transformer (D04)", "Test MAE": transformer["test_mae"],
                 "Seconds": transformer["seconds"]})

    comparison = pd.DataFrame(rows).sort_values("Test MAE").reset_index(drop=True)

comparison.round(1)

In [ ]:
if TORCH_AVAILABLE:
    fig, ax = plt.subplots(figsize=(11, 5))

    ordered = comparison.iloc[::-1]
    colours = ["crimson" if name.startswith("Naive") else
               "seagreen" if "D04" in name else "steelblue"
               for name in ordered["Model"]]
    ax.barh(ordered["Model"], ordered["Test MAE"], color=colours)

    for position, value in enumerate(ordered["Test MAE"]):
        ax.text(value + 6, position, f"{value:.0f}", va="center", fontsize=9)

    ax.set_title("Day-ahead MAE across Part D", fontsize=13, fontweight="bold")
    ax.set_xlabel("MW")
    ax.set_xlim(0, ordered["Test MAE"].max() * 1.12)
    ax.grid(axis="x", linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

The transformer lands **between the convolutional and the recurrent models**: better than both LSTMs,
worse than both CNNs. On a full run it scores 304.2; the workshop configuration reaches 326.7, and its
position in the ranking is the same either way.

It is also by some distance the most expensive thing in Part D. The full run takes around 25 minutes
against the TCN's five and the simple CNN's twenty seconds, for a worse result than either.

The reason is the one set out in section 2. Convolution assumes that nearby positions belong together, and
recurrence assumes that order matters; both are given that structure for free. Attention assumes nothing
and must learn it, and learning a prior that the other architectures are simply handed takes data. Thirty
thousand windows of a single series is a small dataset by the standards of the models that made
transformers famous.

There is direct evidence of under-training in the run itself: validation error was still falling at the
final epoch rather than flattening. More epochs would help, and so would more series. That is the
condition under which transformers win, and it is not the condition here.

**Exercise.** The conclusion above claims the transformer is under-trained, on the evidence that validation error was still falling at the final epoch. Test it: record the validation error at every epoch, then retrain for three times as many and see whether the curve flattens and where the test score ends up. Does more training close the gap to the convolutional models?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="8.-When-a-Transformer-Is-Worth-It">8. When a Transformer Is Worth It</h3>
</div>

| Architecture | Reach for it when | Cost |
|---|---|---|
| **Encoder-only** | A fixed horizon, one shot | Quadratic in window length |
| **Decoder-only** | Long or open-ended horizons | Errors compound step by step |
| **Encoder-decoder** | Rich inputs, known future covariates | The most machinery to get right |

What this notebook established:

**Attention connects any two positions in one step.** That is a genuine architectural advance over walking
a sequence or sliding a filter, and it is why transformers dominate where sequences are long and data is
plentiful.

**Position must be supplied.** A stack of attention layers is permutation equivariant, which the notebook
demonstrates rather than asserts: encoding then permuting and permuting then encoding agree to within
$10^{-7}$. The layer sees a set. For time series, encoding the actual calendar beats encoding the index.

**The freedom costs data.** On this series the transformer lost to a two-layer CNN that trains in twenty
seconds. Attention has to learn the locality that convolution assumes, and there was not enough data to
learn it well.

**Attention weights are not explanations.** The trained model's map is nearly uniform and actively
ignores the most obviously informative lag, while forecasting better than both recurrent models. Do not
read causal stories off attention.

**Quadratic cost is real.** 168 positions means 28,224 comparisons per head per layer. This was the
slowest notebook in the course by a wide margin, which is why the specialised architectures in the next
notebook spend so much of their design effort on avoiding full attention.

---

Every model in Part D so far was built by hand from PyTorch primitives, which is the right way to
understand them and the wrong way to work. The last notebook of this part uses a library of published
architectures instead, and asks whether the state of the art beats what we have already:
[D05 - Specialised Deep Learning Architectures](./D05_Specialised_architectures.ipynb).

**Solutions.** Worked answers to the 2 exercises above, with the reasoning behind them, are in
[D04_Transformers_solutions.ipynb](../solutions/D04_Transformers_solutions.ipynb). Try each one yourself first.
